<a href="https://colab.research.google.com/github/Prashant43226/CUDA-Programming/blob/main/RMSNorm-Profiling-across-CUDA-Triton-JAX/XLA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile rmsnorm.cpp
#include <cuda.h>
#include <cuda_runtime.h>
#include <device_launch_parameters.h>
#include <stdio.h>
#include <stdlib.h>
#include <torch/extension.h>
#include <cooperative_groups.h>

namespace cg = cooperative_groups;

template<typename T>
__global__ void rmsnorm_forward_kernel(const T* x, const T* weight, T* output, float eps, int N){
  int row_idx = blockIdx.x;
  const T* row_x = x + row_idx * N;  //Each row in the input vector will be split into subrows
  //So essentially a tensor of size 4x500 will be split into 4 rows of 500 dimension
  T* row_output = output + row_idx * N;

  float sum_sq = 0.0f;
  //This will aggreate the input values in the first block of each row of lets say size 256(block size)
  for(int i = threadIdx.x; i < N; i += blockDim.x){
    float val = (float)row_x[i];
    sum_sq += val * val;
  }

  // Intra-warp reduction using cooperative groups
  cg::thread_block block = cg::this_thread_block();
  cg::thread_block_tile<32> tile = cg::tiled_partition<32>(block);

  // This will sum the values into each warp's first thread in the first block of each row.
  for(int offset = tile.size() / 2; offset > 0; offset >>= 1){
    sum_sq += tile.shfl_down(sum_sq, offset);
  }

  //Since registers cannot be shared across warps hence need shared memory
  // Shared memory to hold the sums from each warp (max 32 warps for 1024 threads)
  __shared__ float shared_sums[32];
  int warp_id = threadIdx.x / 32;
  int lane_id = threadIdx.x % 32;

  // Each warp's lane 0 writes its local warp sum to shared memory
  if(lane_id == 0){
    shared_sums[warp_id] = sum_sq;
  }
  block.sync();

  //This will sum across all the warp's first thread to a simgle register of first warp's first thread.
  // Inter-warp reduction using the first warp
  if(warp_id == 0){
    sum_sq = (threadIdx.x < (blockDim.x / 32)) ? shared_sums[threadIdx.x] : 0.0f;

    for(int offset = 16; offset > 0; offset >>= 1){
      sum_sq += __shfl_down_sync(0xFFFFFFFF, sum_sq, offset);
    }
  }

  // Re-use a single shared float to broadcast final result
  __shared__ float final_shared_sum;
  if(threadIdx.x == 0){
    final_shared_sum = 1.0f / sqrtf((sum_sq / N) + eps);
  }
  block.sync();

  float rsqrt_rms = final_shared_sum;

  // Compute final normalised output and scale by weight
  for(int i = threadIdx.x; i < N; i += blockDim.x){
    // Added multiplying by the weight[i] parameter as required by RMSNorm
    row_output[i] = (T)((float)row_x[i] * rsqrt_rms * (float)weight[i]);
  }
}

torch::Tensor rmsnorm_forward_cuda(torch::Tensor x, torch::Tensor weight, float eps){
  int N = x.size(-1);
  auto out = torch::empty_like(x);
  int num_token = x.numel() / N;

  // Keep threads to a clean multiple of warp size (256 threads = 8 warps)
  int threads = 256;
  int blocks = num_token;

  AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "rmsnorm_forward_cuda", ([&] {
    rmsnorm_forward_kernel<scalar_t><<<blocks, threads>>>(
      x.data_ptr<scalar_t>(),
      weight.data_ptr<scalar_t>(),
      out.data_ptr<scalar_t>(),
      eps,
      N
    );
  }));
  return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m){
  m.def("rmsnorm_forward", &rmsnorm_forward_cuda, "rmsnorm_forward");
}


Writing rmsnorm.cpp


In [3]:
!ls

rmsnorm.cpp  sample_data


In [11]:
!rm -rf /root/.cache/torch_extensions/

In [18]:
import os
import torch
from torch.utils.cpp_extension import load

# 1. Manually set CUDA_HOME so PyTorch can locate nvcc compiler tools
# Colab always installs CUDA under /usr/local/cuda
os.environ["CUDA_HOME"] = "/usr/local/cuda"

# 2. Read the C++/CUDA code file
with open('/content/rmsnorm.cpp', 'r') as f:
    cuda_source = f.read()

# 3. Write out to an explicit .cu file so PyTorch triggers the CUDA builder
with open('/content/rmsnorm_ext.cu', 'w') as f:
    f.write(cuda_source)

# 4. Compile using the correct arguments
print("Compiling CUDA kernel... (this takes ~1-2 minutes)")
rmsnorm_cuda = load(
    name="rmsnorm_cuda",
    sources=['/content/rmsnorm_ext.cu'],
    extra_cflags=['-O3'],
    verbose=True
)

print("Compilation successful! Module loaded.")

Compiling CUDA kernel... (this takes ~1-2 minutes)
Compilation successful! Module loaded.


In [19]:
# Create test tensors on the GPU
device = 'cuda'
x = torch.randn(4, 256, device=device)       # 4 tokens, hidden dimension 256
weight = torch.ones(256, device=device)       # Weights matching hidden dimension
eps = 1e-5

# Call your custom compiled CUDA function
output = rmsnorm_cuda.rmsnorm_forward(x, weight, eps)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Output Sample (First row, first 5 elements):\n", output[0, :5])


Input shape: torch.Size([4, 256])
Output shape: torch.Size([4, 256])
Output Sample (First row, first 5 elements):
 tensor([ 0.3423,  0.3685,  0.9775, -0.6505, -0.0139], device='cuda:0')
